# EMPRES-i Basic EDA

Self-contained first-pass EDA for `data/structured/empres_i/overview-raw-data_202605211615.csv`. This notebook follows the same structure as the WAHIS and ADIS EDA notebooks while keeping EMPRES-i-specific host, diagnosis source/status, and human impact fields.

## Setup

Shared plotting, display, and helper functions used throughout the notebook.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

pd.options.display.max_columns = 120
pd.options.display.max_rows = 200
pd.options.mode.chained_assignment = None

NA_VALUES = ['NaN', 'nan', '', ';']

def find_data_path(*candidates: str) -> Path:
    paths = [Path(candidate) for candidate in candidates]
    path = next((candidate for candidate in paths if candidate.exists()), None)
    if path is None:
        tried = '\n'.join(str(candidate) for candidate in paths)
        raise FileNotFoundError(f'Could not find source CSV. Tried:\n{tried}')
    return path


def schema_summary(data: pd.DataFrame) -> pd.DataFrame:
    return (
        pd.DataFrame({
            'dtype': data.dtypes.astype(str),
            'missing': data.isna().sum(),
            'missing_pct': data.isna().mean().mul(100).round(1),
            'n_unique': data.nunique(dropna=True),
        })
        .sort_values(['missing_pct', 'n_unique'], ascending=[False, False])
    )


def top_counts(data: pd.DataFrame, column: str, n: int = 15) -> pd.DataFrame:
    return (
        data[column]
        .fillna('Missing')
        .value_counts(dropna=False)
        .head(n)
        .rename_axis(column)
        .reset_index(name='records')
    )


def plot_top_counts(data: pd.DataFrame, specs: list[tuple[str, int, str]], cols: int = 2) -> None:
    rows = -(-len(specs) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 4.8 * rows), squeeze=False)
    for axis, (column, n, title) in zip(axes.ravel(), specs):
        counts = top_counts(data, column, n=n)
        sns.barplot(data=counts, y=column, x='records', ax=axis, color='#4C78A8')
        axis.set_title(title)
        axis.set_xlabel('Records')
        axis.set_ylabel('')
    for axis in axes.ravel()[len(specs):]:
        axis.set_visible(False)
    plt.tight_layout()


def date_coverage(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return pd.DataFrame({
        'date_column': columns,
        'min_date': [data[column].min() for column in columns],
        'max_date': [data[column].max() for column in columns],
        'missing': [data[column].isna().sum() for column in columns],
        'missing_pct': [round(data[column].isna().mean() * 100, 1) for column in columns],
    })


def monthly_counts(data: pd.DataFrame, date_column: str, label: str = 'records') -> pd.DataFrame:
    return (
        data.dropna(subset=[date_column])
        .assign(month=lambda frame: frame[date_column].dt.to_period('M').dt.to_timestamp())
        .groupby('month')
        .size()
        .rename(label)
        .reset_index()
    )


def monthly_counts_by_category(
    data: pd.DataFrame,
    date_column: str,
    category_column: str,
    top_n: int = 6,
    label: str = 'records',
) -> pd.DataFrame:
    top_values = top_counts(data, category_column, n=top_n)[category_column]
    return (
        data[data[category_column].isin(top_values)]
        .dropna(subset=[date_column])
        .assign(month=lambda frame: frame[date_column].dt.to_period('M').dt.to_timestamp())
        .groupby(['month', category_column])
        .size()
        .rename(label)
        .reset_index()
    )


def country_disease_matrix(
    data: pd.DataFrame,
    country_column: str,
    disease_column: str,
    country_n: int = 12,
    disease_n: int = 8,
) -> pd.DataFrame:
    top_countries = top_counts(data, country_column, n=country_n)[country_column]
    top_diseases = top_counts(data, disease_column, n=disease_n)[disease_column]
    filtered = data[data[country_column].isin(top_countries) & data[disease_column].isin(top_diseases)]
    return pd.crosstab(filtered[country_column], filtered[disease_column])

## Load Data

EMPRES-i uses semicolon-delimited CSV exports, decimal commas for coordinate fields, and `dd/mm/yyyy` dates.

In [ ]:
DATA_PATH = find_data_path(
    '../data/structured/empres_i/overview-raw-data_202605211615.csv',
    'data/structured/empres_i/overview-raw-data_202605211615.csv',
)

raw_df = pd.read_csv(
    DATA_PATH,
    sep=';',
    quotechar='"',
    decimal=',',
    na_values=NA_VALUES,
    keep_default_na=True,
    low_memory=False,
)

print(f'Loaded {len(raw_df):,} rows and {raw_df.shape[1]:,} columns from {DATA_PATH}')
raw_df.head()

## Clean Working Copy

Strip the trailing semicolon from `Human.deaths;`, parse dates/numeric fields, and add report delay metrics.

In [ ]:
df = raw_df.copy()
df.columns = df.columns.str.strip().str.rstrip(';')

DATE_COLUMNS = [
    'Observation.date..dd.mm.yyyy.',
    'Report.date..dd.mm.yyyy.',
]
for column in DATE_COLUMNS:
    df[column] = pd.to_datetime(df[column], format='%d/%m/%Y', errors='coerce')

NUMERIC_COLUMNS = ['Latitude', 'Longitude', 'Humans.affected', 'Human.deaths']
for column in NUMERIC_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df['report_delay_days'] = (df['Report.date..dd.mm.yyyy.'] - df['Observation.date..dd.mm.yyyy.']).dt.days

duplicate_summary = pd.Series({
    'duplicate_event_id': df['Event.ID'].duplicated().sum(),
})

display(df.dtypes.to_frame('dtype'))
display(duplicate_summary.to_frame('count'))

## Dataset Overview And Completeness

In [ ]:
overview = pd.DataFrame({
    'rows': [len(df)],
    'columns': [df.shape[1]],
    'countries': [df['Country'].nunique()],
    'diseases': [df['Disease'].nunique()],
    'serotypes': [df['Serotype'].nunique()],
    'animal_types': [df['Animal.type'].nunique()],
    'first_observation_date': [df['Observation.date..dd.mm.yyyy.'].min()],
    'last_observation_date': [df['Observation.date..dd.mm.yyyy.'].max()],
})

display(overview)
display(schema_summary(df))
display(date_coverage(df, DATE_COLUMNS))

## Top Categories

In [ ]:
CATEGORY_COLUMNS = [
    'Disease',
    'Serotype',
    'Country',
    'Subregion',
    'Diagnosis.source',
    'Diagnosis.status',
    'Animal.type',
]

for column in CATEGORY_COLUMNS:
    display(top_counts(df, column, n=20))

In [ ]:
plot_top_counts(
    df,
    [
        ('Country', 15, 'Top Countries'),
        ('Disease', 8, 'Top Diseases'),
        ('Serotype', 12, 'Top Serotypes'),
        ('Animal.type', 12, 'Top Animal Types'),
    ],
)

## Time Coverage And Trends

In [ ]:
events_by_month = monthly_counts(df, 'Observation.date..dd.mm.yyyy.')

sns.lineplot(data=events_by_month, x='month', y='records', marker='o')
plt.title('EMPRES-i Records By Observation Month')
plt.xlabel('Observation month')
plt.ylabel('Records')
plt.xticks(rotation=45)
plt.tight_layout()

In [ ]:
events_by_month_and_disease = monthly_counts_by_category(
    df,
    date_column='Observation.date..dd.mm.yyyy.',
    category_column='Disease',
    top_n=6,
)

sns.lineplot(
    data=events_by_month_and_disease,
    x='month',
    y='records',
    hue='Disease',
    marker='o',
)
plt.title('Top EMPRES-i Diseases Over Time')
plt.xlabel('Observation month')
plt.ylabel('Records')
plt.xticks(rotation=45)
plt.legend(title='Disease', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

## Delay Analysis

In [ ]:
delay_columns = ['report_delay_days']

display(df[delay_columns].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).T)

delay_plot_df = df[df['report_delay_days'].between(0, 365, inclusive='both')]
sns.histplot(data=delay_plot_df, x='report_delay_days', hue='Disease', bins=50, multiple='stack')
plt.title('Reporting Delay Distribution, Capped At 365 Days')
plt.xlabel('Days between observation and report')
plt.ylabel('Records')
plt.tight_layout()

In [ ]:
df.loc[
    df['report_delay_days'].notna(),
    ['Event.ID', 'Country', 'Disease', 'Serotype', 'Observation.date..dd.mm.yyyy.', 'Report.date..dd.mm.yyyy.', 'report_delay_days'],
].sort_values('report_delay_days', ascending=False).head(20)

## Country-Disease Matrix

In [ ]:
matrix = country_disease_matrix(df, 'Country', 'Disease')

sns.heatmap(matrix, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Top EMPRES-i Countries And Diseases')
plt.xlabel('Disease')
plt.ylabel('Country')
plt.tight_layout()

## Geographic Snapshot

In [ ]:
geo_df = df.dropna(subset=['Longitude', 'Latitude']).copy()

display(geo_df[['Longitude', 'Latitude']].describe())

plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=geo_df.sample(min(len(geo_df), 20000), random_state=42),
    x='Longitude',
    y='Latitude',
    hue='Disease',
    alpha=0.35,
    s=12,
    linewidth=0,
)
plt.title('EMPRES-i Event Coordinates, Sampled Up To 20,000 Rows')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(title='Disease', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

## Source-Specific Signals

EMPRES-i includes diagnosis source/status, animal host fields, and human impact fields that are not present in WAHIS or ADIS.

In [ ]:
source_specific = {
    'diagnosis_status_by_disease': pd.crosstab(df['Disease'], df['Diagnosis.status'], dropna=False),
    'diagnosis_source_by_disease': pd.crosstab(df['Disease'], df['Diagnosis.source'], dropna=False),
    'top_admin1': top_counts(df, 'Admin.level.1', 20),
    'top_species': top_counts(df, 'Species', 20),
}

display(source_specific['diagnosis_status_by_disease'])
display(source_specific['diagnosis_source_by_disease'])
display(source_specific['top_admin1'])
display(source_specific['top_species'])

In [ ]:
human_impact = df.loc[
    df[['Humans.affected', 'Human.deaths']].notna().any(axis=1),
    [
        'Event.ID',
        'Disease',
        'Serotype',
        'Country',
        'Observation.date..dd.mm.yyyy.',
        'Humans.affected',
        'Human.deaths',
    ],
].sort_values(['Humans.affected', 'Human.deaths'], ascending=False)

human_impact.head(25)

## Quick Filters

Set any filter to a string value, or leave it as `None` to ignore that filter.

In [ ]:
country_filter = None
disease_filter = None
serotype_filter = None
diagnosis_status_filter = None

filtered = df.copy()
if country_filter:
    filtered = filtered[filtered['Country'].eq(country_filter)]
if disease_filter:
    filtered = filtered[filtered['Disease'].eq(disease_filter)]
if serotype_filter:
    filtered = filtered[filtered['Serotype'].eq(serotype_filter)]
if diagnosis_status_filter:
    filtered = filtered[filtered['Diagnosis.status'].eq(diagnosis_status_filter)]

filtered.sort_values('Report.date..dd.mm.yyyy.', ascending=False).head(50)